In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datetime
import os

import utils.prediction_utils as pred
from utils import dictionaries as dictionary
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR

import camp_tv_new.src.utils.database_utils as db_utils
from camp_tv_new.src.utils.logger import setup_logger, timer_func
import camp_tv_new.src.utils.dictionaries as mydict
from camp_tv_new.src.utils.query_dict import query_dict


In [7]:
## FUNCTIONS

def prep_dataframe(df_bereik: pd.DataFrame) -> pd.DataFrame:


    df = df_bereik[["flight_id", "start_date", "end_date", "advertiser", "product", "daypart",
                    "npo1_grp", "npo2_grp", "npo3_grp", "ster_grp",
                    "ster_1+", "regio_grp"]].reset_index(drop=True)

    df = (df.pivot_table(index=["flight_id", "start_date", "end_date", 'advertiser', 'product'],
                         columns='daypart', fill_value=0.0,
                         values=['npo1_grp', 'npo2_grp', 'npo3_grp',
                                 'ster_grp', 'ster_1+', 'regio_grp'], aggfunc='sum').reset_index())

    df.columns = ["_".join(col) if col[1] else col[0] for col in df.columns.to_flat_index()]

    # Convert start_date and end_date to datetime objects and calculate required values
    df['start_date'] = pd.to_datetime(df['start_date'], format="%d%m%y", errors='coerce')
    df['end_date'] = pd.to_datetime(df['end_date'], format="%d%m%y", errors='coerce')

    # Apply function to create extra columns
    df['timef_start_month'] = df['start_date'].dt.month
    df['timef_end_month'] = df['end_date'].dt.month
    df['timef_start_year'] = df['start_date'].dt.year
    df['timef_end_year'] = df['end_date'].dt.year
    df['timef_start_week'] = df['start_date'].dt.strftime("%V").astype(int)
    df['timef_end_week'] = df['end_date'].dt.strftime("%V").astype(int)
    df['timef_start_day'] = df['start_date'].dt.day
    df['timef_end_day'] = df['end_date'].dt.day
    df['timef_duration_days'] = df.apply(lambda row: (row['end_date'] - row['start_date']).days + 1, axis=1)

    # Calculate total overlap with olympics and WK/EK
    df['timef_olympic_period'] = df.apply(pred.calculate_total_overlap, axis=1, period_list=(dictionary.olympic_period_list))
    df['timef_football_period'] = df.apply(pred.calculate_total_overlap, axis=1, period_list=(dictionary.football_wk_ek_list))

    # # Create meta columns for alcohol and kansspelen
    df['meta_alcohol_adv'] = df.apply(lambda row: pred.is_alcohol(row['advertiser'], row['product'], dictionary.alcohol_kansspelen_dict), axis=1)
    df['meta_kansspelen_adv'] = df.apply(lambda row: pred.is_kansspelen(row['advertiser'], row['product'], dictionary.alcohol_kansspelen_dict), axis=1)

    return df

def create_preprocessor(numerical_cols, categorical_cols, boolean_cols):
    """
    Create a ColumnTransformer with specified transformations for numerical, categorical, and boolean columns.

    Args:
    numerical_cols (list of str): List of numerical column names.
    categorical_cols (list of str): List of categorical column names.
    boolean_cols (list of str): List of boolean column names.

    Returns:
    ColumnTransformer: A preprocessor with the specified transformations.
    """

    # Preprocessing for numerical data: impute missing values and scale
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())])

    # Preprocessing for categorical data: impute missing values and one-hot encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    # Preprocessing for boolean data: impute missing values (if any)
    boolean_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent'))])

    # Combine preprocessing steps
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols),
            ('bool', boolean_transformer, boolean_cols)])

    return preprocessor

def grid_search_models(X_train, y_train, X_test, y_test, models, preprocessor):
    # Initialize variables to store the best model and its score
    best_model = None
    best_score = float('inf')
    best_params = None
    results = []

    # Perform GridSearchCV for each model
    for model_info in models:
        # Create a pipeline with the preprocessor and the model
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model_info['model'])
        ])

        # Create GridSearchCV
        grid_search = GridSearchCV(pipeline, model_info['params'], cv=5, scoring='neg_mean_squared_error', n_jobs=-1, return_train_score=True)
        grid_search.fit(X_train, y_train)
        results.append(pd.DataFrame(grid_search.cv_results_))

        # Get the best score and parameters
        if grid_search.best_score_ < best_score:
            best_score = grid_search.best_score_
            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_

    # Evaluate the best model on the test set
    y_pred = best_model.predict(X_test)
    test_mse = mean_squared_error(y_test, y_pred)

    print(f"Best Model: {best_model}")
    print(f"Best Parameters: {best_params}")
    print(f"Test MSE: {test_mse}")

    return results, best_model, best_params, test_mse

def find_best_model(X_train, y_train, X_test, y_test, model, param_grid, preprocessor, cv=3, scoring='neg_mean_squared_error'):
    """
    """
    # Create a pipeline with the preprocessor and the model
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', model)])

    # Create GridSearchCV object
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring=scoring, n_jobs=-1)

    # Fit the model
    grid_search.fit(X_train, y_train)

    # Best parameters and score
    best_params = grid_search.best_params_
    best_score = -grid_search.best_score_

    # Evaluate the best model on the test set
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    test_mse = mean_squared_error(y_test, y_pred)

    return {
        'best_params': best_params,
        'best_score': best_score,
        'test_mse': test_mse
    }

def query_to_df(query):
    engine = create_sql_alchemy_engine()
    df_query = pd.read_sql_query(query, engine)

    return df_query

def create_sql_alchemy_engine():
    from sqlalchemy import create_engine
    from urllib.parse import quote_plus
    """
    Creates and returns a SQLAlchemy engine for connecting to a SQL Server database.
    """
    quoted = quote_plus(
        "Driver={ODBC+Driver+17+for+SQL+Server};"
        "Server=tcp:stercampsql.database.windows.net;"
        "Database=SterCampNew;"
        "uid=stercampadmin;"
        f"pwd={os.getenv('SQLDB')}"
    )
    alchemy_con = "mssql+pyodbc:///?odbc_connect={}".format(quoted)
    engine = create_engine(alchemy_con)

    return engine

def execute_query_to_df(query):
    engine = create_sql_alchemy_engine()
    SQL_Query = pd.read_sql_query(query, engine)
    df = pd.DataFrame(SQL_Query)

    return df

In [8]:
## VARIABLES TO TWEAK

TEST_SET_DATE = pd.to_datetime('2024-05-01')
CUT_OFF_DATE = '2023-09-01'
TARGET_GROUP = '25-67'

######################################################################################################

query = f'''
    SELECT b.flight_id,
    f.mock_campaign,
    f.advertiser,
    f.product,
    f.start_date,
    f.end_date,
    b.target_group,
    b.campaign_duration,
    b.daypart,
    b.total_grp,
    b.ster_grp,
    b.regio_grp,
    b.npo1_grp,
    b.npo2_grp,
    b.npo3_grp,
    b.[ster_1+]
    FROM [dbo].[tv.bereik] b
    LEFT JOIN [dbo].[tv.flights] f ON b.flight_id = f.flight_id
    where b.ster_grp > 0 and target_group = '{TARGET_GROUP}'
    and start_date >= '{CUT_OFF_DATE}'
    order by start_date
'''

try:
    df = query_to_df(query)
except Exception as e:
    print(f"Error executing query: {str(e)}")


# Prep df
df = prep_dataframe(df)

# Ensure the data is sorted by the start date
df = df.sort_values(by='start_date')

# Split the data into training and test sets based on the cutoff date
train_data = df[df['start_date'] < TEST_SET_DATE]
test_data = df[df['start_date'] >= TEST_SET_DATE]

features = ['npo1_grp_daynight', 'npo1_grp_primetime', 'npo1_grp_total',
            'npo2_grp_daynight', 'npo2_grp_primetime', 'npo2_grp_total',
            'npo3_grp_daynight', 'npo3_grp_primetime', 'npo3_grp_total',
            'regio_grp_daynight', 'regio_grp_primetime', 'regio_grp_total',
            'timef_start_month', 'timef_end_month', 'timef_start_year', 'timef_end_year',
            'timef_start_week', 'timef_end_week', 'timef_start_day', 'timef_end_day',
            'timef_duration_days', 'timef_olympic_period', 'timef_football_period',
            'meta_alcohol_adv', 'meta_kansspelen_adv']

target = 'ster_1+_total'

# Separate features and target for train and test sets
X_train = train_data[features]
y_train = train_data[target]
X_test = test_data[features]
y_test = test_data[target]

# Create the preprocessor. (nu niet echt nodig maar wellicht wel later in de pipeline)
categorical_cols = []
boolean_cols = ['timef_olympic_period', 'meta_alcohol_adv', 'meta_kansspelen_adv', 'timef_football_period']
numerical_cols = [col for col in features if col not in categorical_cols + boolean_cols]
preprocessor = create_preprocessor(numerical_cols, categorical_cols, boolean_cols)


In [9]:
display(df.info())
display(df['start_date'].min())
display(df['start_date'].max())

<class 'pandas.core.frame.DataFrame'>
Index: 7162 entries, 263 to 437
Data columns (total 36 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   flight_id              7162 non-null   object        
 1   start_date             7162 non-null   datetime64[ns]
 2   end_date               7162 non-null   datetime64[ns]
 3   advertiser             7162 non-null   object        
 4   product                7162 non-null   object        
 5   npo1_grp_daynight      7162 non-null   float64       
 6   npo1_grp_primetime     7162 non-null   float64       
 7   npo1_grp_total         7162 non-null   float64       
 8   npo2_grp_daynight      7162 non-null   float64       
 9   npo2_grp_primetime     7162 non-null   float64       
 10  npo2_grp_total         7162 non-null   float64       
 11  npo3_grp_daynight      7162 non-null   float64       
 12  npo3_grp_primetime     7162 non-null   float64       
 13  npo3_gr

None

Timestamp('2023-09-01 00:00:00')

Timestamp('2024-06-27 00:00:00')

In [12]:
df.columns

Index(['flight_id', 'start_date', 'end_date', 'advertiser', 'product',
       'npo1_grp_daynight', 'npo1_grp_primetime', 'npo1_grp_total',
       'npo2_grp_daynight', 'npo2_grp_primetime', 'npo2_grp_total',
       'npo3_grp_daynight', 'npo3_grp_primetime', 'npo3_grp_total',
       'regio_grp_daynight', 'regio_grp_primetime', 'regio_grp_total',
       'ster_1+_daynight', 'ster_1+_primetime', 'ster_1+_total',
       'ster_grp_daynight', 'ster_grp_primetime', 'ster_grp_total',
       'timef_start_month', 'timef_end_month', 'timef_start_year',
       'timef_end_year', 'timef_start_week', 'timef_end_week',
       'timef_start_day', 'timef_end_day', 'timef_duration_days',
       'timef_olympic_period', 'timef_football_period', 'meta_alcohol_adv',
       'meta_kansspelen_adv'],
      dtype='object')

In [10]:
X_train.head(2)

,npo1_grp_daynight,npo1_grp_primetime,npo1_grp_total,npo2_grp_daynight,npo2_grp_primetime,npo2_grp_total,npo3_grp_daynight,npo3_grp_primetime,npo3_grp_total,regio_grp_daynight,...,timef_end_year,timef_start_week,timef_end_week,timef_start_day,timef_end_day,timef_duration_days,timef_olympic_period,timef_football_period,meta_alcohol_adv,meta_kansspelen_adv
263,0.00,7.70,7.70,0.00,1.72,1.72,0.10,1.49,1.59,0.0,...,2023,35,44,1,31,61,0,0,0,0
1922,54.63,48.04,102.66,12.15,24.95,37.10,2.13,7.91,10.04,0.0,...,2023,35,37,1,17,17,0,0,0,0


In [11]:
y_train.head(2)

263      8.55
1922    30.43
Name: ster_1+_total, dtype: float64

In [24]:
df = prep_dataframe(df)

In [13]:
# Define models and their parameters
models = [
    {
        'name': 'RandomForest',
        'model': RandomForestRegressor(),
        'params': {
            'model__n_estimators': [50, 100, 300, 500, 1000],
            'model__max_depth': [None, 10, 20, 30, 40],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__bootstrap': [True, False]
        }
    },
    {
        'name': 'GradientBoosting',
        'model': GradientBoostingRegressor(),
        'params': {
            'model__n_estimators': [50, 100, 300, 500, 1000],
            'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
            'model__max_depth': [3, 5, 7, 10, 12],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__subsample': [0.6, 0.8, 1.0]
        }
    },
    {
        'name': 'SVR',
        'model': SVR(),
        'params': {
            'model__C': [0.1, 1, 10, 100],
            'model__gamma': ['scale', 'auto'],
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__degree': [2, 3, 4],
            'model__epsilon': [0.1, 0.2, 0.5, 1.0]
        }
    },
    {
        'name': 'XGBoost',
        'model': XGBRegressor(objective='reg:squarederror', random_state=42),
        'params': {
            'model__n_estimators': [50, 100, 300, 500, 1000],
            'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
            'model__max_depth': [3, 5, 7, 10, 12],
            'model__gamma': [0, 0.1, 0.2, 0.3, 0.4],
            'model__subsample': [0.6, 0.8, 1.0],
            'model__colsample_bytree': [0.6, 0.8, 1.0],
            'model__min_child_weight': [1, 3, 5],
            'model__reg_alpha': [0, 0.01, 0.1],
            'model__reg_lambda': [1, 1.5, 2]
        }
    }
]


In [27]:
# Define models and their parameters
models = [
    {
        'name': 'RandomForest',
        'model': RandomForestRegressor(),
        'params': {
            'model__n_estimators': [50, 100, 300, 500],
            'model__max_depth': [None, 10, 20, 30]
        }
    },
    {
        'name': 'GradientBoosting',
        'model': GradientBoostingRegressor(),
        'params': {
            'model__n_estimators': [50, 100, 300, 500],
            'model__learning_rate': [0.01, 0.1, 0.2],
            'model__max_depth': [3, 5, 7, 10]
        }
    },
    {
        'name': 'SVR',
        'model': SVR(),
        'params': {
            'model__C': [0.1, 1, 10],
            'model__gamma': ['scale', 'auto'],
            'model__kernel': ['linear', 'rbf']
        }
    },
    {
        'name': 'XGBoost',
        'model': XGBRegressor(),
        'params': {
            'model__n_estimators': [50, 100, 300, 500],
            'model__learning_rate': [0.01, 0.1, 0.2],
            'model__max_depth': [3, 5, 7, 10],
            'model__gamma': [0, 0.1, 0.3],
            'model__subsample': [0.8, 1.0],
            'model__colsample_bytree': [0.8, 1.0]
        }
    }
]

In [ ]:
# Run the grid search function
results, best_model, best_params, test_mse = grid_search_models(X_train, y_train, X_test, y_test, models, preprocessor)

In [31]:
clf.cv_results_

{'mean_fit_time': array([218.8116617 ,   9.74770815, 234.82606378,  16.19960446,
        446.95558381,  12.02929754, 452.9155128 ,  16.62000585,
        286.50995023,  11.52420027, 255.80050094,  23.91386936]),
 'std_fit_time': array([ 5.27721997,  3.5592414 ,  6.82131399,  0.79717668, 19.29146357,
         0.54120742, 23.21809342,  1.59646955, 18.76942178,  0.47437357,
        36.17956214,  1.99174097]),
 'mean_score_time': array([0.73314378, 1.94613931, 0.64252646, 3.00857317, 0.56968985,
        2.45595713, 0.67377274, 3.08167167, 0.5887238 , 2.37273026,
        0.37195377, 2.56199374]),
 'std_score_time': array([0.05590053, 0.61888521, 0.05791082, 0.25230291, 0.05683522,
        0.13741669, 0.22926335, 0.48606939, 0.07244484, 0.11530149,
        0.18028844, 0.25589722]),
 'param_C': masked_array(data=[0.1, 0.1, 0.1, 0.1, 1, 1, 1, 1, 10, 10, 10, 10],
              mask=[False, False, False, False, False, False, False, False,
                    False, False, False, False],
        f

In [14]:
from sklearn.model_selection import RandomizedSearchCV

# Define the XGBRegressor model with early stopping
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# Define the refined parameter grid for RandomizedSearchCV
param_grid = {
    'n_estimators': [400, 450, 500, 550, 600],
    'learning_rate': [0.005, 0.01, 0.015, 0.02],
    'max_depth': [6, 7, 8],
    'gamma': [0.2, 0.3, 0.4],
    'subsample': [0.75, 0.8, 0.85],
    'colsample_bytree': [0.75, 0.8, 0.85],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2]
}

# Set up RandomizedSearchCV with 10-fold cross-validation
random_search = RandomizedSearchCV(estimator=xgb, param_distributions=param_grid, n_iter=100, cv=10, n_jobs=-1, scoring='neg_mean_squared_error', random_state=42)


# Fit the RandomizedSearchCV to the training data with early stopping
random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

# Print the best parameters and best score
print("Best Parameters:", random_search.best_params_)
print("Best Score:", -random_search.best_score_)